# Multithreading & Multiprocessing in Python

Dieses Notebook erklärt **Multithreading** und **Multiprocessing** in Python – mit praktischen Beispielen und typischen Fallstricken
Als Anwendungsbeispiel wird ein  **simpler Multiprocessing-TCP-Server** realisiert, der eine Zeichenkette erwartet und sie wieder zurücksendet (Echo-Server)


## Überblick: Threads vs. Prozesse
Python bietet mehrere Wege, parallel zu arbeiten:

- **Threads (`threading`)**: Mehrere Ausführungsstränge *im selben Prozess* (gemeinsamer Speicher).
- **Prozesse (`multiprocessing`)**: Mehrere *OS-Prozesse* (eigener Speicher pro Prozess).
- **Async (`asyncio`)**: Kooperatives, ereignisgetriebenes Nebenläufigkeitsmodell (nicht Thema dieses Notebooks).

### Faustregeln
- **I/O-lastig** (Netzwerk, Dateien, APIs): **Threads**
- **CPU-lastig** (Bild-/Signalverarbeitung, Simulationen, viele Berechnungen): **Multiprocessing**


## Technisches Detail: Der GIL (Global Interpreter Lock) in Python
In der Standard-Python-Implementierung **CPython** gibt es den **GIL**. Er sorgt dafür, dass **immer nur ein Thread gleichzeitig Python-Bytecode** ausführt
- Das ist eine wohlüberlegte Design-Entscheidung, kein Mangel!
- Andere Plattformen (z.B. Java) erlauben den gleichzeitigen Zugriff auf Code
  - und müssen sich deshalb insbesondere während der Garbage Collection intensiv um Synchronisation und Locks kümmern

Das bedeutet:
- Threads helfen **sehr gut bei I/O** (während ein Thread wartet, kann ein anderer weiterlaufen).
- Threads skalieren **schlecht bei CPU-Workloads**, weil der GIL die echte Parallelität limitiert.
- **Multiprocessing** umgeht den GIL, weil jeder Prozess seinen **eigenen** Interpreter/GIL hat.


* Informationen zur eigenen Python-Installation

In [1]:
import sys, platform
print("Python:", sys.version.split()[0])
print("Implementation:", platform.python_implementation())
print("Platform:", platform.system(), platform.release())


Python: 3.12.1
Implementation: CPython
Platform: Windows 11


## Threads in der Praxis: Blockierende I/O-Zugriffe

- Der blockierende I/O-Zugriff wird einfach mit `time.sleep()` simuliert 


In [2]:
import time
import threading

def fake_io_task(i, delay=0.6):
    time.sleep(delay)
    return f"Task {i} done"

def run_sequential(n=6):
    t0 = time.perf_counter()
    out = [fake_io_task(i) for i in range(n)]
    return out, time.perf_counter() - t0

def run_threads(n=6):
    t0 = time.perf_counter()
    results = [None]*n
    def worker(i):
        results[i] = fake_io_task(i)
    threads = [threading.Thread(target=worker, args=(i,)) for i in range(n)]
    for th in threads: th.start()
    for th in threads: th.join()
    return results, time.perf_counter() - t0

seq_res, seq_dt = run_sequential()
thr_res, thr_dt = run_threads()

print("Sequentiell:", seq_dt, "s")
print("Threads:", thr_dt, "s")
print("Beispielausgabe:", thr_res[:3])


Sequentiell: 3.6247782999416813 s
Threads: 0.6054301999974996 s
Beispielausgabe: ['Task 0 done', 'Task 1 done', 'Task 2 done']


### Detailwissen: Inter-Thread-Kommunikation
- **Lock / RLock**: schützt kritische Abschnitte (Race Conditions vermeiden)
- **Semaphore**: limitiert Parallelität
- **Event**: Ein Signal zwischen Threads
- **Queue**: Thread-sicherer Work-Queue (Producer/Consumer)

In [3]:
from queue import Queue

q = Queue()
for i in range(5):
    q.put(i)

lock = threading.Lock()
total = 0

def consumer():
    global total
    while True:
        try:
            x = q.get_nowait()
        except Exception:
            return
        # kritischer Abschnitt
        with lock:
            total += x
        q.task_done()

threads = [threading.Thread(target=consumer) for _ in range(3)]
for t in threads: t.start()
for t in threads: t.join()

print("Summe:", total)


Summe: 10


## Multiprocessing in der Praxis: CPU-intensive Prozesse

- Hier wird durch eine bewußt ineffiziente Implementierung CPU-Last erzeugt

In [ ]:
import multiprocessing as mp
import time

def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

nums = [34, 34, 34, 34]  # ggf. anpassen, wenn es zu langsam/schnell ist

def run_seq(nums):
    t0 = time.perf_counter()
    out = [fib(n) for n in nums]
    return out, time.perf_counter() - t0

def run_mp_pool(nums):
    t0 = time.perf_counter()
    with mp.Pool(processes=min(len(nums), mp.cpu_count())) as pool:
        out = pool.map(fib, nums)
    return out, time.perf_counter() - t0

seq_out, seq_dt = run_seq(nums)
mp_out, mp_dt = run_mp_pool(nums)

print("Sequentiell:", seq_dt, "s")
print("Multiprocessing Pool:", mp_dt, "s")
print("Outputs equal:", seq_out == mp_out)


### Details: Unterschiede zwischen den Betriebssystemen
`multiprocessing` startet Prozesse je nach OS unterschiedlich:

- **Linux**: standardmäßig `fork`
- **macOS**: zunehmend `spawn` (je nach Python-Version/Settings)
- **Windows**: `spawn`

`spawn` ist sicherer, aber erfordert:
- `if __name__ == "__main__":`-Guard (besonders in Skripten!)
- Funktionen müssen **top-level** definierbar/importierbar sein (kein Lambda im Notebook für Worker-Funktionen)

In Notebooks funktioniert Multiprocessing manchmal eingeschränkt (je nach Umgebung). Für Server-Code ist ein `.py`-Skript oft die robustere Wahl.


In [ ]:
import multiprocessing as mp
print("Default start method:", mp.get_start_method(allow_none=True))
print("Available:", mp.get_all_start_methods())


## Datenaustausch zwischen Prozessen
Prozesse haben getrennten Speicher
- **Der** zentraler Unterschied zum Multithreading
- Der Datenaustausch zwischen Prozessen erfolgt **immer** serialisiert
    - Keine Referenzen
    - Kein direkter Speicherzugriff, immer Umweg über Netzwerk / temporäres Dateisystem
    - Daten werden als Kopie zwischen den Prozessen verschoben

Datenaustausch über:

- **Pipes / Queues** (`multiprocessing.Queue`)
- **Shared Memory** (`multiprocessing.shared_memory`)
- **Manager** (Proxy-Objekte, einfacher aber langsamer)
- **Return Values** via `Pool.map` / `Pool.apply`


In [ ]:
import multiprocessing as mp

def worker(in_q: mp.Queue, out_q: mp.Queue):
    while True:
        item = in_q.get()
        if item is None:
            out_q.put(None)
            return
        out_q.put(item.upper())

def demo_queue():
    in_q, out_q = mp.Queue(), mp.Queue()
    p = mp.Process(target=worker, args=(in_q, out_q))
    p.start()

    for s in ["hallo", "welt", "multiprocessing"]:
        in_q.put(s)
    in_q.put(None)  # stop signal

    outs = []
    while True:
        x = out_q.get()
        if x is None:
            break
        outs.append(x)

    p.join()
    return outs

# In manchen Notebook-Umgebungen kann mp.Process zicken; falls nötig, überspringen.
print(demo_queue())


# Beispiel: Ein einfacher Multiprocessing-TCP-Echo-Server

Eine komplett eigene Implementierung ist in der Realität **nicht notwendig**
- **TCP Server** aus dem Modul `socketserver` sind einsatzbereits

## Server starten
Zum Starten die echo_server.py ausführen

## Test-Client

In [ ]:
import socket

HOST, PORT = "127.0.0.1", 50505

def echo_client(message: str, host=HOST, port=PORT, timeout=3.0) -> str:
    with socket.create_connection((host, port), timeout=timeout) as s:
        s.sendall(message.encode("utf-8"))
        response = s.recv(1024)
    return response.decode("utf-8")

echo_client('Hello Server!')


"received Hello Server! from ('127.0.0.1', 64766) using server process 13852"